In [1]:
import os
import shutil
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# ============================================================
# 1. Train/Test split with grouping to avoid augmentation leakage
# ============================================================
SOURCE_DIR = "/content/drive/MyDrive/Plant_Whisper_AI/processed_dataset_augmented"  # UNBALANCED source — let class_weight handle the imbalance instead of synthetic balancing
OUT_DIR = "/content/dataset_split"  # local Colab disk — Drive can be read-only/slow for many small file copies
CLASSES = ["Cut", "Dry"]
TEST_SIZE = 0.2
SEED = 42

def get_group_id(filename):
    # Strips _augN / _vN suffixes so all variants of one source recording
    # land in the same split (train OR test, never both).
    return re.sub(r'(_aug\d+|_v\d+)?\.\w+$', '', filename)

# Only rebuild the split if it doesn't already exist, so re-running the
# notebook doesn't reshuffle files that are already split.
if not os.path.exists(OUT_DIR):
    for cls in CLASSES:
        files = os.listdir(os.path.join(SOURCE_DIR, cls))
        groups = sorted(set(get_group_id(f) for f in files))

        train_groups, test_groups = train_test_split(
            groups, test_size=TEST_SIZE, random_state=SEED
        )

        for split_name, split_groups in [("train", train_groups), ("test", test_groups)]:
            out_path = os.path.join(OUT_DIR, split_name, cls)
            os.makedirs(out_path, exist_ok=True)
            for f in files:
                if get_group_id(f) in split_groups:
                    shutil.copy(
                        os.path.join(SOURCE_DIR, cls, f),
                        os.path.join(out_path, f)
                    )

        print(f"{cls}: {len(train_groups)} train groups, {len(test_groups)} test groups")
else:
    print(f"{OUT_DIR} already exists — skipping split (delete it first to rebuild).")

Cut: 2640 train groups, 660 test groups
Dry: 6488 train groups, 1622 test groups


In [3]:

# ============================================================
# 2. Data generators
# ============================================================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    width_shift_range=0.05,
    zoom_range=0.05
)
test_datagen = ImageDataGenerator(rescale=1./255)
# test_datagen = ImageDataGenerator()

train_generator = train_datagen.flow_from_directory(
    os.path.join(OUT_DIR, "train"),
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    os.path.join(OUT_DIR, "test"),
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    shuffle=False  # keep order aligned with .classes for evaluation later
)

print("Class indices:", train_generator.class_indices)


Found 9128 images belonging to 2 classes.
Found 2282 images belonging to 2 classes.
Class indices: {'Cut': 0, 'Dry': 1}


In [4]:

# ============================================================
# 3. Class weights (Cut/Dry imbalance fix)
# ============================================================
y_train_labels = train_generator.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_labels),
    y=y_train_labels
)
# # class_weight_dict = dict(enumerate(class_weights))
# # print("Class weights:", class_weight_dict)


In [5]:

# ============================================================
# 4. Build model — MobileNetV2, frozen base
# ============================================================
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3), include_top=False, weights='imagenet'
)


base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',  # default LR = 1e-3
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
]


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:

# ============================================================
# 5. Phase 1 — train classifier head only
# ============================================================
print("--- Phase 1: Training classifier head (base frozen) ---")
print("Base trainable:", base_model.trainable)
print("Optimizer LR:", model.optimizer.learning_rate.numpy())

history = model.fit(
    train_generator,
    epochs=40,
    validation_data=test_generator,
    callbacks=callbacks,
    class_weight=class_weight_dict
)

In [ ]:
model.save("/content/drive/MyDrive/phase1_best.keras")  # save the model itself to Drive
print("Saved phase1_best.keras to Drive")

In [ ]:

# ============================================================
# 6. Phase 2 — fine-tune top layers (BatchNorm kept frozen)
# ============================================================
print("--- Phase 2: Fine-tuning top layers ---")
base_model.trainable = True

# Unfreeze only the last 15 layers — keep the rest frozen
for layer in base_model.layers[:-15]:
    layer.trainable = False

# Critical fix: keep ALL BatchNorm layers frozen even in the unfrozen region,
# otherwise their running stats destabilize on a small dataset.
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_generator,
    epochs=20,
    validation_data=test_generator,
    callbacks=callbacks,
    class_weight=class_weight_dict
)

In [ ]:
model.save("/content/drive/MyDrive/phase2_finetuned.keras")


In [ ]:

# ============================================================
# 7. Evaluation — confusion matrix + classification report
# ============================================================
test_generator.reset()
preds = model.predict(test_generator)
y_pred = (preds > 0.5).astype(int).flatten()
y_true = test_generator.classes

cm = confusion_matrix(y_true, y_pred)
class_names = list(test_generator.class_indices.keys())

print(cm)
print(classification_report(y_true, y_pred, target_names=class_names))

# Save confusion matrix as an image
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names,
    cbar=True
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Plant Whisper AI')
plt.tight_layout()

# cm_path = "/content/drive/MyDrive/Plant_Whisper_AI/confusion_matrix.png"
# plt.savefig(cm_path, dpi=200)
# plt.show()
# print(f"Saved confusion matrix image to {cm_path}")
